In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('../data/processed/model_ready_billz.csv')

In [ ]:
df.shape, df['bill_id'].is_unique, df.isna().sum().sum()

((45641, 31), True, np.int64(0))

45,641 rows and 31 columns. No duplicated bill IDs, no missing values.

In [ ]:
df['outcome'].value_counts(normalize=True)

outcome
1    0.933722
2    0.045047
3    0.021231
Name: proportion, dtype: float64

93.3% of bills died, 4.5% pass their own chamber, and 2.1% become law.

Shows accuracy will be useless as a metric - predicting died every time will score 93%. Will need o use F1, recall, or PR-AUC.

Straitfied splits, class weights?

In [ ]:
df['advanced'] = (df['outcome'] >= 2).astype(int)
df.groupby("commemorative")["advanced"].mean()

commemorative
0    0.060684
1    0.392718
Name: advanced, dtype: float64

commemorative bills become law 39% of the time. 

In [ ]:
df.groupby('sponsor_majority')['advanced'].agg(['mean', 'sum', 'count'])

,mean,sum,count
sponsor_majority,,,
0.0,0.049330,953,19319
1.0,0.078717,2072,26322


: 

When sponsor is in the majority party, bills become law 7.8% of the time on average.

In [ ]:
rates = df.groupby('sponsor_majority')['advanced'].mean()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Minority', 'Majority'], rates.values, width=0.5)
ax.bar_label(bars, labels=[f"{r:.1%}" for r in rates.values], padding=3) # % on top of each bar
ax.set_ylabel('Share of bills that passed their chamber')
ax.set_title('Majority-party sponsor advance bills more often')
ax.set_ylim(0, rates.max() * 1.25) # giving room for labels
ax.spines[['top', 'right']].set_visible(False)
plt.savefig("../figures/eda_majority_advance.png", dpi=150, bbox_inches='tight')
plt.show()